# Chapter 1 — What Are We Actually Doing?

**Book alignment:** PyTorch From First Principles, Chapter 1

**Question this notebook isolates:** Does one forward–backward–update step move `w = 1` toward `3` with gradient `-16`, and does skipping gradient clearing corrupt the very next gradient predictably (`-19.2` instead of `-3.2`)?


In [ ]:
import numpy as np
import torch

torch.manual_seed(0)
np.random.seed(0)


## 1. Hand-derived gradient vs `autograd`

One scalar parameter `w = 1`, input `x = 2`, target `6`, squared error. By hand, `dloss/dw = 2(wx - y)x = -16`. Does PyTorch agree?


In [ ]:
x = torch.tensor(2.0)
target = torch.tensor(6.0)
w = torch.tensor(1.0, requires_grad=True)

prediction = x * w
loss = (prediction - target) ** 2
print(f"prediction={prediction.item()} loss={loss.item()} grad_before={w.grad}")

loss.backward()
hand = 2 * (1.0 * 2.0 - 6.0) * 2.0
print(f"w.grad={w.grad.item()} hand={hand}")


In [ ]:
assert prediction.item() == 2.0
assert loss.item() == 16.0
assert w.grad.item() == hand == -16.0
print("autograd matches the hand derivation: -16.0")


## 2. `backward()` fills `.grad`; only the update moves `w`

`backward()` must not change `w`. A `no_grad` in-place step with `lr = 0.1` must move `w` to `2.6` and the loss to `0.64`, leaving `w` a trainable leaf.


In [ ]:
x = torch.tensor(2.0)
target = torch.tensor(6.0)
w = torch.tensor(1.0, requires_grad=True)

loss = (x * w - target) ** 2
loss.backward()
before = w.item()
print(f"w after backward (must be unchanged): {w.item()}")

with torch.no_grad():
    w -= 0.1 * w.grad

new_loss = ((x * w - target) ** 2).item()
print(f"w={w.item():.4f} new_loss={new_loss:.4f} leaf={w.is_leaf} req={w.requires_grad}")


In [ ]:
assert before == 1.0, "backward() must not move the parameter"
assert abs(w.item() - 2.6) < 1e-6
assert abs(new_loss - 0.64) < 1e-4
assert w.requires_grad and w.is_leaf
print("forward/backward/update are three separate phases")


## 3. Skipping `zero_()` accumulates stale gradients

Without clearing, step-1 `.grad` must equal this step's hand gradient (`-3.2`) plus last step's (`-16.0`), i.e. `-19.2`.


In [ ]:
x = torch.tensor(2.0)
target = torch.tensor(6.0)
w = torch.tensor(1.0, requires_grad=True)

grads = []
for step in range(2):
    loss = (x * w - target) ** 2
    loss.backward()
    grads.append(w.grad.item())
    with torch.no_grad():
        w -= 0.1 * w.grad
    # deliberately no zero_()
print(f"observed grads: {grads}")
print(f"hand step-1 grad: {2 * (2.6 * 2.0 - 6.0) * 2.0}")


In [ ]:
assert grads[0] == -16.0
assert abs(grads[1] - (-3.2 + -16.0)) < 1e-4, grads
assert abs(grads[1] - -19.2) < 1e-4
print("backward() adds to .grad; stale gradients corrupt the update")


## What we earned

Training is three separable phases — forward builds the record, `backward()` populates `.grad`, the `no_grad` update moves the parameter — and a falling first-step loss does not prove the loop is healthy (stale gradients still corrupt step 1+).

Chapter 2 keeps this loop and asks what rules govern the tensors flowing through it: shape, broadcasting, and layout.
